# Notebook 01: Exploring the Cascade Reference Patient Pod

This notebook demonstrates how to use the `cascade-protocol` Python SDK to:

1. Open a Cascade Pod directory
2. Query records by data type
3. Inspect individual records
4. Convert records to a pandas DataFrame

We'll use the **Alex Rivera reference patient pod** — a complete synthetic patient with hypertension, diabetes, asthma, and hyperlipidemia.

**Setup:** `pip install "cascade-protocol[pandas]"`

In [ ]:
import sys
from pathlib import Path

# Add the SDK to the Python path (for development use without pip install)
sdk_root = Path("../src").resolve()
if str(sdk_root) not in sys.path:
    sys.path.insert(0, str(sdk_root))

from cascade_protocol import Pod, CURRENT_SCHEMA_VERSION
print(f"cascade-protocol loaded. Schema version: {CURRENT_SCHEMA_VERSION}")

## 1. Open the Reference Patient Pod

In [ ]:
# Path to the reference patient pod (relative to this notebook)
# Adjust this path if running from a different location
pod_path = Path("../../reference-patient-pod").resolve()

pod = Pod.open(pod_path)
print(f"Pod opened: {pod}")
print(f"\nTurtle files in pod:")
for f in pod.list_files():
    print(f"  {f.relative_to(pod_path)}")

## 2. Query Medications

In [ ]:
medications = pod.query("medications")
print(f"Found {len(medications)} medication(s)\n")

for med in medications:
    status = "active" if med.is_active else "inactive"
    print(f"  [{status}] {med.medication_name}")
    if med.dose:
        print(f"    Dose: {med.dose}")
    if med.frequency:
        print(f"    Frequency: {med.frequency}")
    print(f"    Provenance: {med.data_provenance}")

## 3. Convert to DataFrame

In [ ]:
import pandas as pd

df = medications.to_dataframe()
print(f"DataFrame shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
df[["medication_name", "is_active", "dose", "frequency", "data_provenance"]].head(10)

## 4. Query Conditions

In [ ]:
conditions = pod.query("conditions")
print(f"Found {len(conditions)} condition(s)\n")

cond_df = conditions.to_dataframe()
if not cond_df.empty:
    cols = [c for c in ["condition_name", "status", "onset_date", "condition_class"] if c in cond_df.columns]
    print(cond_df[cols].to_string(index=False))

## 5. Query Vital Signs

In [ ]:
vitals = pod.query("vital-signs")
print(f"Found {len(vitals)} vital sign reading(s)\n")

vitals_df = vitals.to_dataframe()
if not vitals_df.empty:
    cols = [c for c in ["vital_type", "value", "unit", "effective_date", "interpretation"] if c in vitals_df.columns]
    print(vitals_df[cols].to_string(index=False))

## 6. Patient Profile

In [ ]:
profile_set = pod.query("patient-profile")
profile = profile_set.first()

if profile:
    print(f"Patient: {profile.name or 'Unknown'}")
    print(f"Date of birth: {profile.date_of_birth}")
    print(f"Biological sex: {profile.biological_sex}")
    if profile.computed_age:
        print(f"Age: {profile.computed_age} years")
    if profile.blood_type:
        print(f"Blood type: {profile.blood_type}")
else:
    print("No patient profile found in pod")

## 7. Pod Summary

In [ ]:
data_types = [
    "medications", "conditions", "allergies", "lab-results",
    "vital-signs", "immunizations", "insurance",
    "activity", "sleep",
]

print("Pod Summary — Alex Rivera Reference Patient")
print("=" * 40)
for dtype in data_types:
    try:
        rs = pod.query(dtype)
        print(f"  {dtype:<20} {len(rs):>4} record(s)")
    except Exception as e:
        print(f"  {dtype:<20}   -- ({e})")